# Tablas Comparativas de Métricas XAI

Una tabla por corte @k con columnas **AggDiv · IXD · MIL · ECS** y una fila por modelo recomendador y algoritmo XAI.


In [ ]:
import os, glob
import pandas as pd
import numpy as np
from IPython.display import display


In [ ]:
MODO = 'semi'   # 'muestra' | 'semi' | 'completo'
MODELOS = None  # None = detectar todos; o lista: ['FunkSVD', 'ItemKNN']
INCLUIR_LEGACY = False

CWD = os.path.abspath(os.getcwd())
PROJECT_ROOT = CWD
while PROJECT_ROOT and not (
    os.path.isdir(os.path.join(PROJECT_ROOT, 'src')) and
    os.path.isdir(os.path.join(PROJECT_ROOT, 'output'))
):
    parent = os.path.dirname(PROJECT_ROOT)
    if parent == PROJECT_ROOT:
        break
    PROJECT_ROOT = parent

OUTPUT_ROOT = os.path.join(PROJECT_ROOT, 'output')
METRICAS = ['AggDiv', 'IXD', 'MIL', 'ECS']


def descubrir_dirs_evaluacion(modo, modelos=None, incluir_legacy=False):
    dirs = []
    if modelos:
        for modelo in modelos:
            dirs.append((modelo, os.path.join(OUTPUT_ROOT, modelo, f'metricas_evaluacion_{modo}')))
    elif os.path.exists(OUTPUT_ROOT):
        for nombre in sorted(os.listdir(OUTPUT_ROOT)):
            eval_dir = os.path.join(OUTPUT_ROOT, nombre, f'metricas_evaluacion_{modo}')
            if os.path.isdir(eval_dir):
                dirs.append((nombre, eval_dir))

    if incluir_legacy:
        legacy_dir = os.path.join(OUTPUT_ROOT, f'metricas_evaluacion_{modo}')
        if os.path.isdir(legacy_dir):
            dirs.append(('base', legacy_dir))
    return dirs


EVAL_DIRS = descubrir_dirs_evaluacion(MODO, MODELOS, INCLUIR_LEGACY)

if not EVAL_DIRS:
    print(f'??  No hay directorios de evaluaci?n para modo={MODO}')
else:
    for modelo, eval_dir in EVAL_DIRS:
        print(f'? {modelo}: {len(glob.glob(os.path.join(eval_dir, "*.csv")))} CSVs en {eval_dir}')


In [ ]:
# ?? Carga y agregaci?n ????????????????????????????????????????????????????????
def cargar_metricas(eval_dirs):
    registros = []
    for modelo_dir_nombre, base_dir in eval_dirs:
        for path in sorted(glob.glob(os.path.join(base_dir, '*.csv'))):
            basename = os.path.basename(path)
            metrica = next((m for m in METRICAS if f'_{m}_' in basename or f'_{m}.' in basename), None)
            if metrica is None:
                continue
            df = pd.read_csv(path)
            modelo = (
                df['modelo_recomendador'].dropna().iloc[0]
                if 'modelo_recomendador' in df.columns and df['modelo_recomendador'].notna().any()
                else modelo_dir_nombre
            )
            algoritmo = df['algoritmo'].iloc[0] if 'algoritmo' in df.columns else None
            if algoritmo is None:
                continue

            def _mean(col):
                return df[col].dropna().mean() if col in df.columns else np.nan

            if metrica == 'ECS':
                # ECS: una fila por hotel ? agregamos por media
                registros.append(dict(
                    modelo_recomendador=modelo, algoritmo=algoritmo, metrica=metrica,
                    **{f'v{k}': _mean(f'{metrica}{k}') for k in ['', '@1', '@3', '@5']}
                ))
            else:
                # AggDiv, IXD (por usuario) o MIL (fila ?nica)
                for _, row in df.iterrows():
                    alg = row.get('algoritmo', algoritmo)
                    registros.append(dict(
                        modelo_recomendador=modelo, algoritmo=alg, metrica=metrica,
                        **{f'v{k}': row.get(f'{metrica}{k}', np.nan) for k in ['', '@1', '@3', '@5']}
                    ))
    return pd.DataFrame(registros)


df_raw = cargar_metricas(EVAL_DIRS)
if df_raw.empty:
    print('Registros: 0')
else:
    print(
        f'Registros: {len(df_raw)} | '
        f'Modelos: {df_raw["modelo_recomendador"].nunique()} | '
        f'M?tricas: {df_raw["metrica"].unique().tolist()}'
    )


In [ ]:
# ?? Construcci?n de tablas por @k ?????????????????????????????????????????????
def tabla_para_k(df_raw, sufijo):
    col_val = f'v{sufijo}'
    filas = []
    grupos = df_raw[['modelo_recomendador', 'algoritmo']].drop_duplicates().sort_values(['modelo_recomendador', 'algoritmo'])
    for _, grupo in grupos.iterrows():
        modelo = grupo['modelo_recomendador']
        alg = grupo['algoritmo']
        fila = {'Modelo': modelo, 'Algoritmo XAI': alg}
        for m in METRICAS:
            sub = df_raw[
                (df_raw['modelo_recomendador'] == modelo) &
                (df_raw['algoritmo'] == alg) &
                (df_raw['metrica'] == m)
            ]
            fila[m] = round(sub[col_val].mean(), 4) if not sub.empty else np.nan
        filas.append(fila)
    return pd.DataFrame(filas).set_index(['Modelo', 'Algoritmo XAI'])


tablas = {
    'Global (@todos)': tabla_para_k(df_raw, ''),
    '@1':              tabla_para_k(df_raw, '@1'),
    '@3':              tabla_para_k(df_raw, '@3'),
    '@5':              tabla_para_k(df_raw, '@5'),
}
print('Filas:', len(tablas['@1']))


In [ ]:
# ── Visualización minimalista ─────────────────────────────────────────────────
#
# AggDiv, IXD, MIL → verde oscuro = mejor (más alto)
# ECS              → naranja/rojo  = más consistente (más alto = menos personalizado)

CMAPS = {'AggDiv': 'YlGn', 'IXD': 'YlGn', 'MIL': 'YlGn', 'ECS': 'YlOrRd'}
DESC  = {
    'AggDiv': '↑ más diverso',
    'IXD':    '↑ más diverso',
    'MIL':    '↑ más personalizado',
    'ECS':    '↑ más consistente',
}

TABLA_STYLES = [
    {'selector': 'table',
     'props': [('border-collapse', 'collapse'), ('font-family', 'Inter, sans-serif'),
               ('font-size', '13px'), ('width', 'auto')]},
    {'selector': 'caption',
     'props': [('font-size', '14px'), ('font-weight', '600'), ('text-align', 'left'),
               ('padding-bottom', '8px'), ('color', '#222')]},
    {'selector': 'th',
     'props': [('background', '#1a1a2e'), ('color', '#fff'), ('padding', '8px 16px'),
               ('text-align', 'center'), ('font-weight', '500'), ('letter-spacing', '0.03em')]},
    {'selector': 'th.row_heading',
     'props': [('text-align', 'left'), ('min-width', '160px')]},
    {'selector': 'td',
     'props': [('padding', '6px 16px'), ('text-align', 'center'), ('border-bottom', '1px solid #f0f0f0')]},
    {'selector': 'tr:hover td',
     'props': [('filter', 'brightness(0.93)')]},
]


def mostrar_tabla(tabla, titulo):
    cols = [m for m in METRICAS if m in tabla.columns]
    styler = (tabla[cols]
              .style
              .set_caption(titulo)
              .format('{:.4f}', na_rep='—')
              .set_table_styles(TABLA_STYLES))
    for col in cols:
        styler = styler.background_gradient(cmap=CMAPS[col], subset=[col], axis=0)
    display(styler)


for titulo, tabla in tablas.items():
    mostrar_tabla(tabla, f'Métricas XAI — {titulo}')
    print()   # espacio entre tablas


In [ ]:
# ?? Exportar ??????????????????????????????????????????????????????????????????
OUTPUT_DIR = os.path.join('..', '..', 'output', f'visualizacion_tablas_{MODO}')
os.makedirs(OUTPUT_DIR, exist_ok=True)

nombres = ['tabla_metricas_global.csv', 'tabla_metricas_at1.csv',
           'tabla_metricas_at3.csv',    'tabla_metricas_at5.csv']

for nombre, tabla in zip(nombres, tablas.values()):
    tabla.to_csv(os.path.join(OUTPUT_DIR, nombre))

print(f'? Tablas exportadas a {OUTPUT_DIR}')
